<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](mlcourse.ai) – دورة مفتوحة للتعلم الآلي 
### <center> المؤلف: تاتيانا كوداسوفا، ODS Slack @kudasova
    
## <center> هل ستنجح في المواعدة السريعة؟



### الجزء الأول. وصف مجموعة البيانات والميزات
في عالم اليوم المزدحم، يبدو أن العثور على شريك رومانسي والتعارف معه يستغرق وقتًا أطول من أي وقت مضى. كنتيجة ل, تحول العديد من الأشخاص إلى المواعدة السريعة كحل يسمح للمرء بلقاء عدد كبير من الشركاء المحتملين والتفاعل معهم في فترة زمنية قصيرة. كما اتضح، يجد بعض الأشخاص شريكًا بالفعل خلال هذه الأحداث، والبعض الآخر لا يرى أبدًا أيًا من الأشخاص الذين التقوا بهم هناك. في هذا المشروع، نستكشف ما يميز أولئك الذين يواعدون بعد المواعدة السريعة ("بيانات سريعة جيدة أو ناجحة") عن أولئك الذين لا يفعلون ذلك ("بيانات سريعة سيئة")، ونحاول التنبؤ بما إذا كان الشخص سيكون جيدًا أم سيئًا. لذا يمكنك أن تجرب بنفسك هل ستكون المواعدة السريعة مفيدة لك أم لا.تم جمع البيانات من 552 مشاركًا في أحداث المواعدة التجريبية السريعة من 2002 إلى 2004 (إجمالي 21 حدثًا). وكان المشاركون طلابًا في كليات الدراسات العليا والمهنية في جامعة كولومبيا. وقد تم تجنيدهم من خلال مجموعة من رسائل البريد الإلكتروني والمنشورات المنشورة في جميع أنحاء الحرم الجامعي، وتم توزيعها من قبل مساعدي الباحثين. من أجل الاشتراك في حدث المواعدة السريعة، كان على الطلاب المهتمين التسجيل في موقع ويب عبر الإنترنت حيث قاموا بالإبلاغ عن أسمائهم وعناوين بريدهم الإلكتروني وأكملوا استبيان ما قبل الحدث. خلال الأحداث، سيكون للحاضرين "موعد أول" لمدة أربع دقائق مع كل مشارك آخر من الجنس الآخر. في نهاية الدقائق الأربع، سُئل المشاركون عما إذا كانوا يرغبون في رؤية موعدهم مرة أخرى. وطُلب منهم أيضًا تقييم موعدهم بناءً على ست سمات: الجاذبية، والإخلاص، والذكاء، والمرح، والطموح، والاهتمامات المشتركة. في الصباح التالي لحدث المواعدة السريعة، تم إرسال بريد إلكتروني إلى المشاركين يطلب منهم إكمال استبيان المتابعة عبر الإنترنت. أكمل معظم المشاركين في المواعدة السريعة استبيان المتابعة هذا من أجل الحصول على التطابقات الخاصة بهم. عند استلام ردود استبيان المتابعة، تم إرسال بريد إلكتروني إلى المشاركين لإبلاغهم بنتائج المطابقة.
أيضًا بعد 3-4 أسابيع من حدث المواعدة السريعة، طُلب من المشاركين ملء استبيان مرة أخرى. وإجابة واحدة من هذا الاستطلاع نأخذها كهدف هنا - `date_3` - ​​ما إذا كان الشخص قد ذهب إلى موعد مع واحدة على الأقل من مباريات المواعدة السريعة الخاصة به. جميع ميزاتنا ستكون من استطلاعات الرأي السابقة للحدث، حيث أن هدفنا هو التنبؤ بالأشخاص الذين لم يسبق لهم حضور مثل هذه الأحداث. لمزيد من التفاصيل، راجع ملف speed-dating-data-key.doc الخاص بقاموس البيانات ومفتاح الأسئلة. مفتاح السؤال والبيانات التي تجدونها على [Kaggle]: (https://www.kaggle.com/annavictoria/speed-dating-experiment).
### الجزء الثاني. تحليل البيانات الأولية


In [ ]:
# importing packages
%matplotlib inline
import pandas as pd
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import recall_score

In [ ]:
# importing data
df = pd.read_csv('data/Speed Dating Data.csv', encoding="ISO-8859-1") 
df.head()

In [ ]:
# our data has 8378 rows and 195 columns
df.shape


أولاً، في هذا المشروع، نتوقع نجاح المواعدة السريعة، ونحن نستخدم فقط المعلومات التي تم جمعها قبل حدوث المواعدة السريعة، لأن المعلومات التي تم جمعها بعد المواعدة السريعة سوف تسرب توقعاتنا. لذلك نقوم بحذف الأعمدة التي تحتوي على المعلومات التي تم جمعها بعد المواعدة السريعة، باستثناء المتغير المستهدف.


In [ ]:
columns_after = df.ix[:, 'satis_2':].columns.values
target = df['date_3']
df = df.drop(columns_after, axis=1)
df['target'] = target
df.shape


حسنًا، لقد حصلنا على أعمدة أقل بكثير. ثانيًا، تحتوي مجموعة البيانات هذه على عدة صفوف لكل مشارك (نفس عدد الشركاء الذين التقى بهم خلال المواعدة السريعة). جزء واحد من المتغيرات كلها متماثلة في تلك الصفوف لمشارك واحد - فهي خاصة بالمشارك، ونحن مهتمون بها. الجزء الآخر يدور حول كيفية تقييم المشارك لشريكه الحالي في المواعدة السريعة، وأولئك الذين لا نهتم بهم. دعونا ننظر إلى هذا:


In [ ]:
df[df['iid'] == 1] # participant with her unique iid = 1


لنقم بتصفية جميع المتغيرات التي لا تتعلق بالمشارك فقط، أي التي تحتوي على أكثر من قيمة فريدة لكل مشارك:


In [ ]:
uniques_dict = df[df['iid'] == 1].nunique()
non_uniques = [k for k, v in uniques_dict.items() if v > 1]
df_person = df.drop(non_uniques, axis=1)
df_person.drop_duplicates(subset='iid', keep='first', inplace=True)
df_person.shape


الآن حصلنا على صفوف أقل بكثير. وأيضًا، نحتاج فقط إلى البيانات إذا لم يكن المتغير المستهدف مفقودًا، لذلك دعونا نقوم بتصفية ذلك.


In [ ]:
df_person = df_person[pd.notna(df_person['target'])]
df_person.shape


إنه لأمر محزن، ولكن لدينا نصف البيانات فقط. دعونا ننظر إلى ما بقي لدينا.


In [ ]:
df_person.head()

In [ ]:
df_person.describe().T


لدينا هنا صف معين يحتوي على الكثير من زمالة المدمنين المجهولين (لم يهتم المشارك بالاستبيانات)، لذلك قمنا بحذفه فقط.


In [ ]:
df_person[df_person['iid']==136]

In [ ]:
df_person = df_person[df_person['iid']!=136]
df_person.shape


وسنقوم أيضًا بإسقاط الميزات التي تحتوي على أكثر من 20% من NAs، والتي لا تبدو موثوقة بالنسبة لي.


In [ ]:
na_count = df_person.isna().sum()
too_many_nans = [k for k, v in na_count.items() if v > df_person.shape[0]*0.20]
too_many_nans

In [ ]:
df_person = df_person.drop(too_many_nans, axis=1)
df_person.shape


والآن دعونا نلقي نظرة على الميزات التي لا تحتوي على الكثير من زمالة المدمنين المجهولين ونحاول ملء زمالة المدمنين المجهولين هناك.


In [ ]:
na_count = df_person.isna().sum()
na_count[na_count > 0]

أما بالنسبة لـ `zipcode` (الرمز البريدي للمنطقة التي نشأ فيها المشارك)، أعتقد أن هذه المعلومات مبالغ فيها، لأن لدينا بالفعل متغير `from` (إجابة على ميزة الانتظار التي قد تكون مفيدة: `foreign` = 1 إذا كان الطالب في الأصل ليس من الولايات المتحدة الأمريكية. 


In [ ]:
df_person['foreign'] = (df_person['zipcode'].isna()) | (df_person['zipcode'] == '0')
df_person['foreign'] = df_person['foreign'].astype(int)


دعونا نتحقق مما إذا كان كل شيء على ما يرام وأن البيانات نظيفة.


In [ ]:
df_person[['iid', 'from', 'zipcode', 'foreign']].tail()


لا، أنها ليست نظيفة. سأقوم بتنظيفه باليد. لم يستغرق الأمر الكثير من الوقت، لأن مجموعة البيانات صغيرة.


In [ ]:
to_foreign = [16, 43, 44, 45, 46, 48, 74, 80, 83, 84, 89, 90, 100, 116, 133, 139, 153, 171, 183, 189, 198, 221, 224, 242, 247, 253, 280, 317, 360, 398,  401, 404, 469, 486, 530, 546, 548, 549, 552]
from_foreign = [187, 220, 244, 278, 322, 328, 331, 333, 336, 462, 526]
foreign_list = []
for index, row in df_person.iterrows():
    if row['iid'] in to_foreign:
        foreign = 1
    elif row['iid'] in from_foreign:
        foreign = 0
    else:
        foreign = row['foreign']
    foreign_list.append(foreign)
df_person['foreign'] = foreign_list


يمكننا الآن حذف العمود `zipcode`.


In [ ]:
df_person = df_person.drop('zipcode', axis=1)


الآن دعونا نلقي نظرة على الصفوف المفقودة `career_c` ونحاول الإشارة إليها من `career`.


In [ ]:
df_person[['iid', 'career', 'career_c']][df_person['career_c'].isna()]

In [ ]:
df_person.loc[:10, 'career_c'] = 1 # 1 stands for 'Law'


أما بقية الأعمدة التي تحتوي على NAs فهي رقمية، وسنملأ NAs هناك بالمتوسطات.


In [ ]:
num_na = df_person.columns[df_person.isna().any()].tolist()
df_person = df_person.fillna(df_person.median())
na_count = df_person.isna().sum()
na_count[na_count > 0]


تهانينا! ليس لدينا المزيد من القيم المفقودة!
لا يزال لديك شيء آخر للقيام به. في الاستطلاعات، تم طرح الأسئلة حول سمات الأشخاص بشكل مختلف في موجات مختلفة من التجربة. الموجات 6-9: يرجى تقييم أهمية السمات التالية في تاريخ محتمل على مقياس من 1-10 (1= غير مهم على الإطلاق، 10= مهم للغاية):
الموجات 1-5، 10-21: لديك 100 نقطة للتوزيع بين السمات التالية - قم بإعطاء المزيد من النقاط لتلك السمات الأكثر أهمية في تاريخ محتمل، ونقاط أقل لتلك السمات الأقل أهمية في تاريخ محتمل.  مجموع النقاط يجب أن يساوي 100.
لذلك نحن بحاجة إلى توسيع نطاق البيانات لتجنب هذا الاختلاف.


In [ ]:
what_looks_for =['attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1']
for index, row in df_person.iterrows():
    if any(row[what_looks_for] > 10):
        best = max(row[what_looks_for])
        df_person.loc[index, what_looks_for] = row[what_looks_for]*10/best
df_person[what_looks_for].head()

In [ ]:
what_opposite_sex_looks_for =['attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1', 'shar2_1']
for index, row in df_person.iterrows():
    if any(row[what_opposite_sex_looks_for] > 10):
        best = max(row[what_opposite_sex_looks_for])
        df_person.loc[index, what_opposite_sex_looks_for] = row[what_opposite_sex_looks_for]*10/best
df_person[what_opposite_sex_looks_for].head()


 أخيرًا، سنقوم بإسقاط أعمدة أرقام التعريف والأعمدة المتعلقة بحدث المواعدة السريعة.


In [ ]:
df_person = df_person.drop(['iid', 'id', 'idg', 'wave', 'condtn', 'round', 'position', 'match_es'], axis=1)


الآن دعونا نلقي نظرة على أنواع البيانات لدينا.


In [ ]:
df_person.info()


دعونا نجعل من المتغيرات الثنائية نوع ثنائي.


In [ ]:
binary = ['gender', 'foreign', 'target']
df_person[binary] = df_person[binary].astype(bool)


دعونا نرى كم عدد المشاركين الذين لديهم موعد واحد على الأقل كل شهر بعد المواعدة السريعة.


In [ ]:
df_person['target'].value_counts()

In [ ]:
df_person['target'].value_counts(normalize=True)

لذا فإن 168 من أصل 262 شخصًا (أي حوالي 64%) لم يكن لديهم أي مواعيد بعد المواعدة السريعة. دعونا نرى ما إذا كانت المتغيرات الثنائية لدينا: الجنس والحالة الأجنبية لها قيمة هنا.


In [ ]:
df_person.groupby(['target'])['gender', 'foreign'].agg([np.mean, np.std, np.min, np.max])


نرى أن الجنس لا يؤثر على المتغير المستهدف، ولكن قد يؤثر الوضع الأجنبي. دعونا نتحقق مما إذا كان هذا فرقًا ذا دلالة إحصائية باستخدام اختبار مربع كاي لاستقلال المتغيرات في جدول الطوارئ.


In [ ]:
g, p, dof, expctd = chi2_contingency(pd.crosstab(df_person['foreign'], df_person['target']).values)
p


القيمة P هي 0.63 وهي أكثر من 0.05 لذا لا يمكننا رفض فرضيتنا الصفرية. في مثل هذه المجموعة الصغيرة من البيانات، من الصعب إثبات أهمية الميزة إحصائيًا.



الآن دعونا نجعل بياناتنا الفئوية لها نوع فئوي.


In [ ]:
categories={1: "Black / African American", 
            2: "European/Caucasian-American",
            3: 'Latino/Hispanic American', 
            4: 'Asian/Pacific Islander/Asian-American',
            5: 'Native American', 
            6: 'Other'}
df_person.race = df_person.race.apply(lambda x: categories[x])

In [ ]:
categories={1: 'Law',  
            2: 'Math',
            3: 'Social Science, Psychologist',
            4: 'Medical Science, Pharmaceuticals, and Bio Tech',
            5: 'Engineering',
            6: 'English/Creative Writing/ Journalism ',
            7: 'History/Religion/Philosophy', 
            8: 'Business/Econ/Finance',
            9: 'Education, Academia',
            10: 'Biological Sciences/Chemistry/Physics',
            11: 'Social Work',
            12: 'Undergrad/undecided',
            13: 'Political Science/International Affairs', 
            14: 'Film',
            15: 'Fine Arts/Arts Administration',
            16: 'Languages',
            17: 'Architecture',
            18: 'Other'}
df_person.field_cd = df_person.field_cd.apply(lambda x: categories[x])

In [ ]:
categories={1: 'Seemed like a fun night out',
            2: 'To meet new people',
            3: 'To get a date',
            4: 'Looking for a serious relationship',
            5: 'To say I did it',
            6: 'Other'}
df_person.goal = df_person.goal.apply(lambda x: categories[x])

In [ ]:
categories={1: 'Several times a week',
            2: 'Twice a week',
            3: 'Once a week',
            4: 'Twice a month',
            5: 'Once a month',
            6: 'Several times a year',
            7: 'Almost never'}
df_person.date = df_person.date.apply(lambda x: categories[x])

In [ ]:
categories={1: 'Several times a week',
            2: 'Twice a week',
            3: 'Once a week',
            4: 'Twice a month',
            5: 'Once a month',
            6: 'Several times a year',
            7: 'Almost never'}
df_person.go_out = df_person.go_out.apply(lambda x: categories[x])

In [ ]:
categories={1: 'Lawyer',
            2: 'Academic/Research',
            3: 'Psychologist',
            4: 'Doctor/Medicine',
            5: 'Engineer',
            6: 'Creative Arts/Entertainment',
            7: 'Banking/Consulting/Finance/Marketing/Business/CEO/Entrepreneur/Admin',
            8: 'Real Estate', 
            9: 'International/Humanitarian Affairs',
            10: 'Undecided',
            11: 'Social Work',
            12: 'Speech Pathology',
            13: 'Politics',
            14: 'Pro sports/Athletics',
            15: 'Other',
            16: 'Journalism',
            17: 'Architecture'}
df_person.career_c = df_person.career_c.apply(lambda x: categories[x])


العلاقة بين المتغيرات الأخرى والهدف سوف نقوم بتصورها وشرحها في الفقرة التالية!
### الجزء 3. تحليل البيانات المرئية الأولية
أولاً، دعونا نرى ما إذا كان العمر يلعب دورًا هنا.


In [ ]:
sns.boxplot(x="target", y="age", data=df_person)


على الرغم من أن عمر البيانات الناجحة أعلى في المتوسط، إلا أن الفرق صغير.
الآن دعونا نرى ما إذا كان العرق مهم.


In [ ]:
sns.countplot(y='race', hue='target', data=df_person)


حسنًا، مقارنة بجميع الأجناس الأخرى، يميل اللاتينيون إلى أن يكونوا أكثر نجاحًا في البيانات السريعة. لكن المجموعة الفرعية من اللاتينيين صغيرة جدًا بحيث لا يمكن استخدامها كميزة منفصلة. دعونا نلقي نظرة على مجال الدراسة المشفر.


In [ ]:
sns.countplot(y='field_cd', hue='target', data=df_person)


في الرسم البياني أعلاه نرى أن هناك بيانات سريعة أكثر نجاحًا بين أولئك الذين يدرسون التاريخ أو الدين أو الفلسفة أو اللغة الإنجليزية أو الكتابة الإبداعية أو الصحافة. ولكن كما هو الحال مع السباقات، فإن المجموعة الفرعية من بيانات السرعة الجيدة هنا صغيرة جدًا.


In [ ]:
sns.countplot(y='goal', hue='target', data=df_person)


ليس من المستغرب أن أولئك الذين يبحثون عن علاقة جدية يميلون إلى الحصول على مواعيد بعد المواعدة السريعة. لكن، للأسف، هذه الفئة صغيرة جدًا لذا لا يمكن استخدامها كميزة منفصلة.


In [ ]:
sns.countplot(y='date', hue='target', data=df_person)


في الرسم البياني أعلاه وعلى الرسم البياني أدناه، لا أرى أي فرق بين توزيع بيانات السرعة الجيدة والسيئة.

In [ ]:
sns.countplot(y='go_out', hue='target', data=df_person)


هل يؤثر الناقل على نجاح البيانات السريعة؟


In [ ]:
sns.countplot(y='career_c', hue='target', data=df_person)


من المثير للدهشة أم لا، يميل الأخصائيون الاجتماعيون إلى أن يكونوا أفضل البيانات السريعة. ولكن مرة أخرى، هذه المجموعة من المشاركين صغيرة جدًا.
هل تؤثر هواية الأشخاص على سلوكهم في المواعدة السريعة؟


In [ ]:
interests = ['target', 
             'sports', 'tvsports', 'exercise', 'dining', 'museums', 'art', 'hiking', 'gaming', 'clubbing', 'reading', 'tv', 'theater', 'movies', 'concerts', 'music', 'shopping', 'yoga']
df_long = pd.melt(df_person[interests], "target", var_name="interests", value_name="points")
sns.catplot(y="interests", hue="target", x="points", data=df_long, kind="box")


بالنسبة لمعظم الهوايات، لا يحدث ذلك. ولكن إذا كنت تستمتع بمشاهدة التلفاز، فإنك تميل إلى أن تكون سريعًا في المواعدة، وإذا كنت تستمتع بالتمارين الرياضية - فمن المحتمل أن تصبح شخصًا سريعًا بشكل أفضل.
الآن دعونا نتفحص ميزة `exphappy` - الإجابة على السؤال: "بشكل عام، على مقياس من 1 إلى 10، ما مدى السعادة التي تتوقعها أن تكون مع الأشخاص الذين تقابلهم أثناء حدث المواعدة السريعة؟"


In [ ]:
sns.boxplot(x="target", y="exphappy", data=df_person)


من المثير للاهتمام أن البيانات ذات السرعة الجيدة تتوقع أن تكون أقل سعادة من بيانات السرعة السيئة. لذلك، كلما كنت أقل تفاؤلاً بشأن المواعدة السريعة، كلما كان ذلك أفضل! 
ماذا عن متغير أهمية العرق؟


In [ ]:
sns.boxplot(x="target", y="imprace",  data=df_person)


بالنسبة للبيانات ذات السرعة الجيدة، يميل السباق إلى أن يكون أقل أهمية من البيانات ذات السرعة السيئة. أما بالنسبة لأهمية الدين (انظر أدناه)، فلا يبدو أنها تحدث أي فرق.


In [ ]:
sns.boxplot(x="target", y="imprelig", data=df_person)


أما بالنسبة للصفات التي يبحث عنها المشارك، فهناك بعض الاختلاف هناك. يميل أصحاب البيانات ذات السرعة السيئة إلى البحث عن المزيد من الجاذبية والصدق والذكاء والطموح والاهتمامات المشتركة. للحصول على بيانات جيدة السرعة، المتعة أكثر أهمية.


In [ ]:
what_looks_for = ['target', 
                  'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1']
df_long = pd.melt(df_person[what_looks_for], "target", var_name="what_looks_for", value_name="points")
sns.catplot(y="what_looks_for", hue="target", x="points", data=df_long, kind="box")


هناك أيضًا اختلاف في السمات التي يعتقد المشارك أن الجنس الآخر يبحث عنها في الموعد. يميل أصحاب البيانات السريعة السيئة إلى توقع أن تكون الجاذبية والصدق والذكاء والمرح والاهتمامات المشتركة أكثر أهمية بالنسبة للجنس الآخر. يراهن أصحاب البيانات السريعة الجيدة على الطموح.


In [ ]:
what_opposite_sex_looks_for = ['target', 
                  'attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1', 'shar2_1']
df_long = pd.melt(df_person[what_opposite_sex_looks_for], "target", var_name="what_opposite_sex_looks_for", value_name="points")
sns.catplot(y="what_opposite_sex_looks_for", hue="target", x="points", data=df_long, kind="box")


ومن المثير للاهتمام أن تقدير المشاركين لذاتهم لنفس السمات ليس مهمًا على الإطلاق!


In [ ]:
how_you_measure_up = ['target', 
                  'attr3_1', 'sinc3_1', 'intel3_1', 'fun3_1', 'amb3_1']
df_long = pd.melt(df_person[how_you_measure_up], "target", var_name="how_you_measure_up", value_name="points")
sns.catplot(y="how_you_measure_up", hue="target", x="points", data=df_long, kind="box")

لنقم بإنشاء إطار بيانات جديد باستخدام الميزات التي وجدناها قد تؤثر على الهدف ثم نرى ما إذا كانت هذه الميزات مرتبطة أم لا.


In [ ]:
binary_cols = ['foreign']
numeric_cols = ['age', 'imprace', 'exercise', 'tv', 'exphappy', 
                'attr1_1', 'sinc1_1', 'intel1_1', 'fun1_1', 'amb1_1', 'shar1_1',
                'attr2_1', 'sinc2_1', 'intel2_1', 'fun2_1', 'amb2_1', 'shar2_1']
df_full = pd.concat([df_person[binary_cols], df_person[numeric_cols]], axis=1)
target = df_person['target']
sns.heatmap(df_full.corr())


حسنًا، نرى أن معظم سمات الشخص ترتبط ببعضها البعض، ولكن ليس بدرجة كافية لحذفها.
### الجزء الرابع. الأنماط والرؤى وخصائص البيانات
مجموعة البيانات صغيرة حقًا، ولهذا السبب كان من المهم جدًا أن نقوم بتنظيف بياناتنا وفحصها بعناية فائقة. نحن بحاجة إلى تقليل عدد الميزات حتى لا نبالغ في نموذجنا. كما هو موضح في هذه [الورقة](https://bmcmedinformdecismak.biomedcentral.com/articles/10.1186/1472-6947-12-8#CR31_460)، بالنسبة للميزات المرتبطة، فمن الأفضل أن تكون الميزات $\sqrt N$ هي حجم العينة. لقد وجدنا ميزات قد تؤثر على المتغير المستهدف: الحالة الأجنبية، العمر، أهمية العرق، حب التلفزيون، ممارسة الرياضة، توقع السعادة، السمات التي يبحث عنها المشارك في تواريخه (إجمالي 6)، السمات التي يعتقد المشارك أن الجنس الآخر يبحث عنها في تواريخه (إجمالي 6) - 18 سمة. 
### الجزء 5. اختيار المقاييس
في هذا المشروع نتعامل مع مشكلة التصنيف مع فئتين غير متوازنتين إلى حد كبير. إذا توقعنا لشخص ما أنه شخص سيء السرعة في حالة أنه في الواقع شخص جيد السرعة - وهذا سلبي كاذب - فهو أكثر إيذاءً لنا. لذا فإن تقليل السلبيات الكاذبة هو أكثر أهمية بالنسبة لنا، ولهذا السبب نختار مقاييس الاستدعاء.
### الجزء 6. اختيار النموذج
في حالة مجموعة البيانات الصغيرة، هناك فرصة كبيرة لحدوث نقص في التجهيز أو زيادة في التجهيز. لتجنب التجهيز الزائد، يوصى باستخدام نموذج بسيط مع عدد منخفض من المعلمات الفائقة وتقليل عدد الميزات. لتجنب النقص في التجهيز، يجب ألا يكون النموذج بسيطًا جدًا. لهذا السبب نختار مصنف SVM.
### الجزء السابع. المعالجة المسبقة للبياناتنقوم هنا بتوسيع نطاق ميزاتنا الرقمية باستخدام StandardScaler حتى يعمل SVC بشكل صحيح.


In [ ]:
scaler = StandardScaler()
df_full[numeric_cols] = scaler.fit_transform(df_full[numeric_cols])
df_full.head()


### الجزء 8. التحقق من صحة وتعديل المعلمات الفائقة للنموذج
قبل إجراء التحقق المتبادل، سنقوم بعمل عينة اختبار لاختبار نموذجنا النهائي. نحن نفترض أن بياناتنا مستقلة عن الوقت، ونجري تقسيمًا عشوائيًا طبقيًا بنسبة 10% من البيانات.


In [ ]:
df_full.shape, target.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_full, 
                                                    target, test_size=0.1, random_state=17, stratify=target.values)
X_train.shape, X_test.shape


للتحقق من الصحة، نظرًا لأن مجموعة البيانات لدينا صغيرة والفئات غير متوازنة، فإننا نستخدم تقسيمًا طبقيًا بمقدار 10 أضعاف.


In [ ]:
cv = StratifiedKFold(n_splits=10, random_state=17)


نختار مصنف SVM بأوزان متوازنة. مقاييس التسجيل لدينا هي التذكير. نقوم بضبط معلمة التنظيم `C` والنوع `kernel`.


In [ ]:
clf = SVC(class_weight='balanced', gamma='scale', random_state=17)
params = {
    'C': [0.01, 0.1, 1, 10],
    'kernel': ('linear', 'poly', 'rbf', 'sigmoid')
}
gscv = GridSearchCV(clf, params, scoring='recall', cv=cv)
gscv.fit(X_train, y_train)
gscv.best_params_


أفضل المعلمات لدينا هي: النواة الخطية مع C = 0.01. لنقم الآن بإجراء تحقق متقاطع آخر مع نطاق أقرب للمعلمة `C`.


In [ ]:
clf = SVC(kernel='linear', class_weight='balanced', gamma='scale', random_state=17)
params = {
    'C': [0.005, 0.01, 0.02]
}
gscv = GridSearchCV(clf, params, scoring='recall', cv=cv)
gscv.fit(X_train, y_train)
gscv.best_params_


لا يزال C = 0.01 هو الأفضل، لذلك نستخدمه لتدريب مجموعة بيانات القطار بأكملها.


In [ ]:
clf = SVC(C=0.01, kernel='linear', class_weight='balanced', gamma='scale', random_state=17)
clf.fit(X_train, y_train)


### الجزء 9. إنشاء ميزات جديدة ووصف لهذه العملية
في الجزء السابع قمنا بإنشاء ميزة جديدة - الوضع الأجنبي. دعونا نرى ما إذا كان من المهم لنموذجنا.


In [ ]:
clf.coef_, X_train.columns.values


وكما نرى من أوزان ميزاتنا، فهو ليس المتغير الأقل أهمية، لذا فهو مفيد. ومع ذلك، فإن الميزات الأكثر قيمة هي أهمية العرق، وحب التلفزيون، وتوقع أهمية الاهتمامات المشتركة للجنس الآخر، وأهمية الاهتمامات المشتركة للمشارك.



### الجزء العاشر. رسم منحنيات التدريب والتحقق من الصحة


In [ ]:
from sklearn.model_selection import learning_curve

plt.figure()
train_sizes, train_scores, test_scores = learning_curve(
    clf, X_train, y_train, cv=cv, scoring='recall', random_state=17)

train_scores_mean = np.mean(train_scores, axis=1)
train_scores_std = np.std(train_scores, axis=1)
test_scores_mean = np.mean(test_scores, axis=1)
test_scores_std = np.std(test_scores, axis=1)
plt.grid()

plt.xlabel("Number of samples")
plt.ylabel("Metrics")
plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                 train_scores_mean + train_scores_std, alpha=0.1,
                 color="r")
plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                 test_scores_mean + test_scores_std, alpha=0.1, color="g")
plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
         label="Training curve")
plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
         label="Validation curve")

plt.legend(loc="best")
plt.show()

يمكن لمنحنى التحقق الخاص بنا أن يتقارب مع منحنى التدريب إذا تمت إضافة المزيد من حالات التدريب. لذا فإن إضافة المزيد من البيانات من المحتمل جدًا أن يساعد هنا. لدينا أخطاء عالية إلى حد ما في كل من منحنيات التدريب والتحقق من الصحة، مما يعني أن نموذجنا به انحياز عالي (أي غير مناسب). لدينا أيضًا تباين كبير حول منحنى التحقق.
### الجزء 11. التنبؤ بالعينات الاختبارية أو المحتجزة
قمنا ببناء مجموعة بيانات القطار والاختبار في الجزء 8. هنا نقوم بتدريب نموذجنا على مجموعة بيانات القطار ونحصل على درجة الاستدعاء الخاصة بنا. 


In [ ]:
y_pred = clf.predict(X_test)
recall_score(y_test, y_pred)


وهي أقل قليلاً من درجة التحقق المتقاطع والتي تبلغ حوالي 0.65 كما هو موضح في الصورة من الجزء السابق. لكن هذا أمر مفهوم، حيث نعرض للنموذج بيانات جديدة.
### الجزء 12. الاستنتاجات
لقد صممنا نموذجًا بدرجة استدعاء = 0.6 مما يعني أنه قادر على العثور على ما يقرب من 60% من بيانات السرعة الجيدة في البيانات. على الرغم من أنه أفضل من لا شيء، هناك مجال كبير للتحسين هنا. أولاً، سيكون من الأفضل لو كان لدينا المزيد من البيانات. ثانيًا، قد نقوم بعمل أفضل في هندسة الميزات، ونجد بعض ارتباطات الميزات، ونتخذ نموذجًا أكثر تعقيدًا. كانت المهمة صعبة لأنه لم تكن هناك ميزات ترتبط كثيرًا بالهدف، ومن الصعب فصل تلك الفئات. لذلك قد نحاول بعض خوارزميات تقليل الأبعاد. 
أما الآن، فلدينا نموذج بسيط يمكنك من خلاله التنبؤ بما إذا كنت ستنجح في المواعدة السريعة أم لا. ومن ميزتها أنها تحتوي على تفسير جميل. تقول أنه كلما قل اهتمامك بعرق شريكك، قل حبك لمشاهدة التلفزيون، وقل اهتمامك بمشاركة الاهتمامات، وقل توقعك من شريكك أن يهتم به، كلما زاد النجاح الذي قد تحققه في المواعدة السريعة. كن متسامحًا وشاهد التلفاز بشكل أقل!